 # Bridging GA runs to HPC jobs

 ## Getting Base Cmd Working
 Okay!  So I go overwhelmed trying to remember how to get the HPC up and running when i have a CMD line I know works.  So here is a simple notebook to help me remember how to do that!

 The golden cmdline I know works is:

 ```
 python3 /home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/compare_tng_to_spam.py --function exp_1 --target_id 62000000190717 --input balance_base --input2 test --output /home/mbo2d/galStuff/data/spam_tuner_exp_1
 ```

Go ahead and run this in a cmdline after calling
```
./scripts/sing-wr.sh
```


You'll probably get a File exists and exists early, add --overwrite to the cmdline to fix that.

    ``` 
    python3 /home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/compare_tng_to_spam.py --function exp_1 --target_id 62000000190717 --input balance_base --input2 test --output /home/mbo2d/galStuff/data/spam_tuner_exp_1 --overwrite
    ```
It'll auto detect it's running on a machine with X cores and auto allocates those cores for the run. And it should output something like this. 

```
[2026-01-09 14:00:46,748 - INFO - compare_tng_to_spam.func_experiment_1]: SPAM Tuner Setup Complete
[2026-01-09 14:00:47,780 - INFO - parallel_queue_manager._run_local_processes]: Local Manager: Progress: [=                   ] 2/30
[2026-01-09 14:00:48,274 - INFO - parallel_queue_manager._run_local_processes]: Local Manager: Progress: [====================] 30/30 - Queue Complete
[2026-01-09 14:00:49,063 - INFO - parallel_queue_manager._run_local_processes]: Local Manager: Progress: [====================] 25/25 - Queue Complete
[2026-01-09 14:00:49,065 - INFO - compare_tng_to_spam.func_experiment_1]: GA Run Complete
```


## Singularity Wrapper

Next, we need this to run in a singularity wrapper.   We were already running in a singularity wrapper above, but now we need to make a script to launch it.  Here is what the script at this location should looking like

loc: /home/mbo2d/galStuff/scripts/sing-wr.bash
```
#!/bin/bash

cmd="./$1"
echo "sing-wr.sh: $cmd"

# Load singularity
module load singularity

# singularity exec --nv /home/mbo2d/images/simr-image-0.sif $cmd
singularity exec --nv /home/mbo2d/images/babbage-beta-2.sif $cmd

```

Note, All it does is take the argument it's given, loads singularity in the HPC environment, and runs it within the singularity exec command.  Try it now, outside a singularity shell.   Run

```
./scripts/sing-wr.bash "python3 /home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/compare_tng_to_spam.py --function exp_1 --target_id 62000000190717 --input balance_base --input2 test --output /home/mbo2d/galStuff/data/spam_tuner_exp_1 --overwrite"
```

It'll output something very similar to above, but now it's running in singularity!

```
[2026-01-09 14:39:21,589 - INFO - compare_tng_to_spam.func_experiment_1]: Results saved to /home/mbo2d/galStuff/data/spam_tuner_exp_1/Exp0_target=62000000190717__score=balance_base__ga=test_results.png
INFO:    Terminating squashfuse_ll after timeout
INFO:    Timeouts can be caused by a running background process
```

## Slurm Command.

Next we need to create a SLURM script to launch this on the HPC.  See below script that will create the file for you!  We'll remove the overwrite flag since we want to keep results from multiple runs.

Let's save that to disk for you!

In [2]:
import os
slurm_batch_wrapper_txt = """#!/bin/bash
#SBATCH -J spam_exp_1-2
#SBATCH -p research-cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --exclusive
#SBATCH -t 12:00:00
#SBATCH -o /home/mbo2d/galStuff/slurm_batch_logs/spam_exp_1-2/%x_%j.out
#SBATCH -e /home/mbo2d/galStuff/slurm_batch_logs/spam_exp_1-2/%x_%j.err

set -euo pipefail

tid="${1:?target_id required}"
score="${2:?score key required}"
ga="${3:?ga key required}"
out="${4:?output dir required}"

module load singularity
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 NUMEXPR_NUM_THREADS=1

CMD="python3 /home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/compare_tng_to_spam.py \
  --function exp_1 --target_id ${tid} --input ${score} --input2 ${ga} \
  --output ${out}"

echo "Running: $CMD"
singularity exec --nv /home/mbo2d/images/babbage-beta-2.sif bash -lc "$CMD"
"""

batch_dir = '/home/mbo2d/galStuff/slurm_batch_scripts/'

# create dir if it doesn't exist
os.makedirs( batch_dir, exist_ok=True )


# Let's save the batch script
file_name = "spam_ga_tuner_exp_1-2_wrapper.sbatch"
batch_path = os.path.join( batch_dir, file_name )
with open( batch_path, 'w' ) as f:
    f.write( slurm_batch_wrapper_txt )

## Run the SLURM Script

Try running this cmd line to launch the SLURM script

```
sbatch /home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch 62000000190717 balance_base test /home/mbo2d/galStuff/data/spam_tuner_exp_1-2
```

You should see output like this. `Submitted batch job 44879`
Use `squeue -u mbo2d` to see your job in the queue!
And check folder `/home/mbo2d/galStuff/slurm_batch_logs/` for a new log file!
NOTE: My logger outputs everything to the .err file so check that file for output!
```
[2026-01-09 14:55:26,482 - INFO - compare_tng_to_spam.func_experiment_1]: Running Experiment 1
...
[2026-01-09 14:55:27,894 - INFO - optimization_manager.__init__]: Initializing PYGAD Manager
...
[2026-01-09 14:55:28,966 - INFO - parallel_queue_manager._run_local_processes]: Local Manager: Progress: [==                  ] 3/30
[2026-01-09 14:55:29,558 - INFO - parallel_queue_manager._run_local_processes]: Local Manager: Progress: [====================] 30/30 - Queue Complete
[2026-01-09 14:55:30,386 - INFO - parallel_queue_manager._run_local_processes]: Local Manager: Progress: [====================] 25/25 - Queue Complete
...
[2026-01-09 14:55:37,768 - INFO - compare_tng_to_spam.func_experiment_1]: Results saved to /home/mbo2d/galStuff/data/spam_tuner_exp_1/Exp0_target=62000000190717__score=balance_base__ga=test_results.png

```

---


Next!  Let's create a test Python scrip to run 4 targets in test mode to verify everything is working!  See code below that creates that script!....  NVM, apparently i'm missing most of those particles... Top 10!

In [3]:
# Get list of available particles in data file

# list all files in dir
data_dir = '/home/mbo2d/galStuff/data/tng-data'
import os
files = os.listdir(data_dir)

# Filter to files that contain 'target-archive' in name
files = [ f for f in files if 'target-archive' in f]

# only grab the target id in the name
top_ids = [ f.split('-')[2].split('.')[0] for f in files ]
# print(top_ids)

In [4]:
test_job_submit_txt = """import os, subprocess

top_id_file = '/home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/tng-targets/moi-4-top-ratings.txt'
sbatch_script = '/home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch'
output_dir = '/home/mbo2d/galStuff/data/spam_tuner_exp_1-2'

score = 'balance_base'
ga = 'test'

# Let's try this.
data_dir = '/home/mbo2d/galStuff/data/tng-data'
files = os.listdir(data_dir)

# Filter to files that contain 'target-archive' in name
files = [ f for f in files if 'target-archive' in f]

# only grab the target id in the name
top_ids = [ f.split('-')[2].split('.')[0] for f in files ]

# Take first X
for tid in top_ids[:4]:
    cmd = [
        'sbatch',
        sbatch_script,
        str(tid),
        score,
        ga,
        output_dir,
    ]

    print("Submitting:", " ".join(cmd))
    subprocess.run(cmd, check=True)
"""

batch_dir = '/home/mbo2d/galStuff/slurm_batch_scripts/'

# create dir if it doesn't exist
os.makedirs( batch_dir, exist_ok=True )


# Let's save the batch script
file_name = "spam_ga_tuner_exp_1-2_test_job_submit.py"
batch_path = os.path.join( batch_dir, file_name )
with open( batch_path, 'w' ) as f:
    f.write( test_job_submit_txt )

Run 
```
python3 ./slurm_batch_scripts/spam_ga_tuner_exp_1-2_test_job_submit.py
```
and you'll see something like this!
```
[mbo2d@c2 galStuff]$ python3 ./slurm_batch_scripts/spam_ga_tuner_exp_1-2_test_job_submit.py 
Submitting: sbatch /home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch 51000000196622 balance_base balance /home/mbo2d/galStuff/data/spam_tuner_exp_1-2
Submitted batch job 44892
Submitting: sbatch /home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch 54000000045003 balance_base balance /home/mbo2d/galStuff/data/spam_tuner_exp_1-2
Submitted batch job 44893
Submitting: sbatch /home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch 49000000112167 balance_base balance /home/mbo2d/galStuff/data/spam_tuner_exp_1-2
Submitted batch job 44894
Submitting: sbatch /home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch 37000000126111 balance_base balance /home/mbo2d/galStuff/data/spam_tuner_exp_1-2
Submitted batch job 44895
```

In [5]:
test2_submit_txt = """import os, subprocess

top_id_file = '/home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/tng-targets/moi-4-top-ratings.txt'
sbatch_script = '/home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch'
output_dir = '/home/mbo2d/galStuff/data/spam_tuner_exp_1-2'

score = 'balance_base'
ga_list = [ 'balance', 'explore', 'exploit' ]

# Let's try this.
data_dir = '/home/mbo2d/galStuff/data/tng-data'
files = os.listdir(data_dir)

# Filter to files that contain 'target-archive' in name
files = [ f for f in files if 'target-archive' in f]

# only grab the target id in the name
top_ids = [ f.split('-')[2].split('.')[0] for f in files ]
tid = 62000000190717

# Take first X
for ga in ga_list:
    cmd = [
        'sbatch',
        sbatch_script,
        str(tid),
        score,
        ga,
        output_dir,
    ]

    print("Submitting:", " ".join(cmd))
    subprocess.run(cmd, check=True)
"""

batch_dir = '/home/mbo2d/galStuff/slurm_batch_scripts/'

# create dir if it doesn't exist
os.makedirs( batch_dir, exist_ok=True )


# Let's save the batch script
file_name = "spam_ga_tuner_exp_1-2_test2_job_submit.py"
batch_path = os.path.join( batch_dir, file_name )
with open( batch_path, 'w' ) as f:
    f.write( test2_submit_txt )

In [6]:
main_job_submit_txt = """import os, subprocess

top_id_file = '/home/mbo2d/galStuff/galaxyJSPAM/IllustrisTNG/tng-targets/moi-4-top-ratings.txt'
sbatch_script = '/home/mbo2d/galStuff/slurm_batch_scripts/spam_ga_tuner_exp_1-2_wrapper.sbatch'
output_dir = '/home/mbo2d/galStuff/data/spam_tuner_exp_1-2'

score = 'balance_base'
ga_list = [ 'balance', 'explore', 'exploit' ]

# Let's try this.
data_dir = '/home/mbo2d/galStuff/data/tng-data'
files = os.listdir(data_dir)

# Filter to files that contain 'target-archive' in name
files = [ f for f in files if 'target-archive' in f]

# only grab the target id in the name
top_ids = [ f.split('-')[2].split('.')[0] for f in files ]

for tid in top_ids:
    for ga in ga_list:
        cmd = [
            'sbatch',
            sbatch_script,
            str(tid),
            score,
            ga,
            output_dir,
        ]

        print("Submitting:", " ".join(cmd))
        subprocess.run(cmd, check=True)
"""

batch_dir = '/home/mbo2d/galStuff/slurm_batch_scripts/'

# create dir if it doesn't exist
os.makedirs( batch_dir, exist_ok=True )


# Let's save the batch script
file_name = "spam_ga_tuner_exp_1-2_main_job_submit.py"
batch_path = os.path.join( batch_dir, file_name )
with open( batch_path, 'w' ) as f:
    f.write( main_job_submit_txt )

# Reviewing Results

Let's only look at the best results.

In [3]:
"""
File: spam_tuner_exp_2.py
Author: Matthew Ogden
Email: ogdenm12@gmail.com
Github: mbogden
Created: 2026-01-09

Description: Code to run Experiment 2 of SPAM tuner.  Using a genetic algorithm to optimize SPAM hyperparameters for multiple targets.

References: 
- Sections of this code were written with the assistance of ChatGPT made by OpenAI.
- TNG50 Simulation Data
    - Nelson, D. et al. (2015). The Illustris Simulation: Public Data Release. Astronomy and Computing, 13, 12–37. https://doi.org/10.1016/j.ascom.2015.09.003
    - Nelson, D. et al. (2019). First results from the TNG50 simulation: Galactic outflows driven by supernovae and black hole feedback. Monthly Notices of the Royal Astronomical Society, 490(3), 3234–3261. https://doi.org/10.1093/mnras/stz2306
    - Pillepich, A. et al. (2019). First results from the TNG50 simulation: The evolution of stellar and gaseous discs across cosmic time. Monthly Notices of the Royal Astronomical Society, 490(3), 3196–3233. https://doi.org/10.1093/mnras/stz2338

"""
# ================================ IMPORTS ================================ #
import hashlib, os, sys, logging, time
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from mpi4py import MPI
from copy import deepcopy
import glob

# Add main project directory to import project modules
try:
    SCRIPT_DIRECTORY = os.path.dirname(os.path.realpath(__file__))
    PROJECT_DIRECTORY = os.path.dirname(SCRIPT_DIRECTORY)
    sys.path.append(PROJECT_DIRECTORY)

except:
    # Assume we are running from the script directory
    SCRIPT_DIRECTORY = os.getcwd()
    PROJECT_DIRECTORY = os.path.dirname(SCRIPT_DIRECTORY)
    
    # Also assume you might be in galStuff directory
    work_dir = os.getcwd()
    if 'galStuff' == work_dir.split('/')[-1]:
        PROJECT_DIRECTORY =  work_dir + '/galaxyJSPAM'
        sys.path.append(PROJECT_DIRECTORY)

# Import project modules
import IllustrisTNG.tng_functions as tf
import IllustrisTNG.convert_tng_to_spam as cts
import IllustrisTNG.compare_tng_to_spam as compare_tng_to_spam
import utilities.parallel_queue_manager as qm
# import Optimization_Methods.optimization_manager as om
import Simulator.new_simulator as sim
import utilities.general_utility as gu

# ================================ GLOBALS ================================ #
script_args, logger = gu.initialize_environment( module_name='spam_tuner' )
buildEnv = False  # For testing and development purposes only

# Set NumPy print options to reduce the precision to 3 decimal places
logger.debug("Imports Done")

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-s898dc_1 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
[2026-01-12 15:52:14,676 - INFO - optimization_manager.<module>]: Imported Loaded


In [4]:
# Development Cell:  This should only be present in the development Jupyter Notebook version of this script
buildEnv = True

from importlib import reload

# Warn user if NOT in jupyter notebook, require a yes/no response to continue or not
if "ipykernel" not in sys.modules:
    msg = "Development Environment Detected.  Do you want to continue?..."

    # Get user input
    while True:
        response = input(f"{msg} (y/n): ").lower()
        if response in ['y', 'yes']:
            break
        elif response in ['n', 'no']:
            print("Exiting...")
            sys.exit()
        else:
            print("Invalid input. Please enter 'y' or 'n'.")



In [5]:
if buildEnv:
    #Let's load some target data
    all_targets_df = pd.read_pickle('galaxyJSPAM/IllustrisTNG/tng-targets/moi-4-targets.pkl')
    print( all_targets_df.shape)
    # print(all_targets_df.columns)

(1268, 111)


In [6]:
if buildEnv and False:
    top_df = all_targets_df[ all_targets_df['moi_4_rating'] == 4]
    print(f"Rank 5 Tidal Features: {top_df.shape}")

    results_dir = "/home/mbo2d/galStuff/data/spam_tuner_exp_1-2"
    file_name = "cmp-tng-spam-exp_target=%d_score=balance_base_ga=%s_results.png"
    ga_types = [ 'explore', 'balance', 'exploit']

    # Create a 3 x 7 subplot for visualization
    fig, axes = plt.subplots(19, 3, figsize=(20, 80))

    i = 0
    for idx, row in top_df.iterrows():

        t_id = int(row['p_SubhaloIDRaw'])

        print( f"Results for {t_id}")

        for j, ga_type in enumerate(ga_types):
            # print( f"Target ID: {t_id}, GA Type: {ga_type}")
            result_file = file_name % (t_id, ga_type)
            result_path = os.path.join(results_dir, result_file)
            print(f"\t- {os.path.exists(result_path)}: - {result_file}")

            # load and add image to subplot
            if os.path.exists(result_path):
                img = plt.imread(result_path)
                ax = axes[i, j]
                ax.imshow(img)
                ax.axis('off')
                ax.set_title(f"{ga_type}: {t_id}", fontsize=8)
        
        i += 1

        # break


    # Lo

In [7]:

best_results = {}
if buildEnv:
    top_df = all_targets_df[ all_targets_df['moi_4_rating'] == 4]
    print(f"Rank 5 Tidal Features: {top_df.shape}")

    results_dir = "/home/mbo2d/galStuff/data/spam_tuner_exp_1-2"
    file_name = "cmp-tng-spam-exp_target=%d_score=balance_base_ga=%s_results.csv"
    ga_types = [ 'explore', 'balance', 'exploit' ]

    i = 0

    for idx, row in top_df.iterrows():

        t_id = int(row['p_SubhaloIDRaw'])
        print( f"Target ID: {t_id}")

        best_results[ t_id ] = {}

        for ga_type in ga_types:

            result_file = file_name % (t_id, ga_type)
            result_path = os.path.join(results_dir, result_file)
            # print(f"\t- {os.path.exists(result_path)}: - {result_file}")

            if os.path.exists(result_path):
                tmp_df = pd.read_csv(result_path)
                # print( tmp_df.head()    )
                # print( tmp_df.shape)
                
                # Find the lowest 'error' value
                best_row = tmp_df.loc[tmp_df['error'].idxmin()]
                best_results[t_id][ga_type] = best_row.to_dict()

                print(f"\t- Best row for {ga_type}: {best_row.to_dict()}")

        i += 1

        # break

    # Lo

Rank 5 Tidal Features: (44, 111)
Target ID: 51000000196622
	- Best row for explore: {'lnl': 0.0314873101884369, 'm_halo': 28.60777087172361, 'r_scale': 27.25432287996069, 'error': 0.0706519486045205}
	- Best row for balance: {'lnl': 0.027468873432649, 'm_halo': 25.591139231283805, 'r_scale': 25.783620835132886, 'error': 0.0707050378325384}
	- Best row for exploit: {'lnl': 0.0280410385801578, 'm_halo': 26.055408233645014, 'r_scale': 26.167534041103305, 'error': 0.0704061488864542}
Target ID: 54000000045003
Target ID: 49000000112167
Target ID: 37000000126111
Target ID: 32000000118358
Target ID: 38000000107405
Target ID: 39000000115251
Target ID: 32000000114609
Target ID: 51000000229119
	- Best row for explore: {'lnl': 0.0637086380918484, 'm_halo': 2.3731080488716736, 'r_scale': 10.836442955239892, 'error': 3.725881374581739}
	- Best row for balance: {'lnl': 0.0907579834282792, 'm_halo': 4.979155843780647, 'r_scale': 14.497926762231574, 'error': 3.539130469031511}
	- Best row for exploit:

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

results_dir = "/home/mbo2d/galStuff/data/spam_tuner_exp_1-2"
pts_name = "spam-pts_target=%d_ga-type=%s_n-pts=%d.npz"
n_pts = 100000

def f(t_id):

    try:
        target = compare_tng_to_spam.SPAM_TNG_Tuner( target_id=t_id )
        target._gen_spam_collision_parameters( radius_scale = 7, softening_scale = 0.1 )
    except Exception as e:
        print(f"Failed:\n{e}")
        return None
    # expensive work

    for ga_type, result in best_results[t_id].items():
        pts_file = pts_name % (t_id, ga_type, n_pts)
        pts_path = os.path.join(results_dir, pts_file)

        if os.path.exists(pts_path):
            print(f"Skipping existing file: {pts_path}")
            continue

        # Here you would create and save the new Particles based on 'result'
        _, pts = target.run_spam_particles( spam_param=result, n_pts=n_pts )
        
        # Save the generated Particles to file
        np.savez_compressed(pts_path, pts=pts)
        print(f"Saved Particles to: {pts_path}")

        # Regen collision parameters with larger radius
    return t_id


if buildEnv and True:
    # Let's create new Particles for each simulation  
    inputs = []                   
    for t_id, ga_results in best_results.items():
        inputs.append(t_id)
        # break

    # inputs = list(range(10))

    with ProcessPoolExecutor() as ex:  # defaults to os.cpu_count() workers
        futures = [ex.submit(f, x) for x in inputs]
        results = [fu.result() for fu in as_completed(futures)]
    print(results)



[2026-01-12 15:52:29,953 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,954 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,958 - INFO - convert_tng_to_spam.extract_tng_target]: Extracting Target Data for: 51000000196622
[2026-01-12 15:52:29,954 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,954 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,955 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,955 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,955 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,956 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,956 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:29,954 - INFO - co

Failed:
'SPAM_TNG_Tuner' object has no attribute 'pts'

[2026-01-12 15:52:31,550 - INFO - convert_tng_to_spam.extract_tng_target]: Loaded Data: (21, 111)


[2026-01-12 15:52:31,554 - INFO - convert_tng_to_spam.extract_tng_target]: Loaded Data: (21, 111)
[2026-01-12 15:52:31,555 - INFO - convert_tng_to_spam.load_pts_data]: Loading from Target Archive: /home/mbo2d/galStuff/data/tng-data/target-archive-51000000196622.h5
[2026-01-12 15:52:31,555 - INFO - convert_tng_to_spam.load_pts_data]: Loading from Target Archive: /home/mbo2d/galStuff/data/tng-data/target-archive-90000000374020.h5
[2026-01-12 15:52:31,562 - INFO - convert_tng_to_spam.load_pts_data]: Loading TNG Particles Data: 74000000302102
[2026-01-12 15:52:31,569 - INFO - convert_tng_to_spam.load_pts_data]: Loading TNG Particles Data: 81000000325459
[2026-01-12 15:52:31,571 - INFO - convert_tng_to_spam.load_pts_data]: Loading TNG Particles Data: 32000000118358
[2026-01-12 15:52:31,571 - INFO - convert_tng_to_spam.load_pts_data]: Loading TNG Particles Data: 78000000319495
[2026-01-12 15:52:31,573 - INFO - convert_tng_to_spam.load_pts_data]: Loading TNG Particles Data: 82000000437637
[20

Failed:
Unable to open file (truncated file: eof = 250609664, sblock->base_addr = 0, stored_eof = 354641767)


[2026-01-12 15:52:31,704 - INFO - convert_tng_to_spam.load_pts_data]: Loading TNG Particles Data: 83000000442011
[2026-01-12 15:52:31,709 - INFO - convert_tng_to_spam.__init__]: Initializing TNG Target Data
[2026-01-12 15:52:31,711 - INFO - convert_tng_to_spam.extract_tng_target]: Extracting Target Data for: 71000000424510


... (25 outputs left out)

[2026-01-12 15:52:54,626 - INFO - convert_tng_to_spam._gen_splines_hermite]: Interpolating path with Hermite cubic splines...


[2026-01-12 15:52:54,648 - INFO - convert_tng_to_spam._gen_moment_of_closest_approach]: Generating Most Recent Moment of Closest Approach (MOCA) Info...
[2026-01-12 15:52:54,705 - INFO - convert_tng_to_spam._gen_spam_collision_parameters]: Generating SPAM collision parameters for target: 78000000319495
[2026-01-12 15:52:54,722 - INFO - convert_tng_to_spam.standardize_collision_parameters_v2]: Standardizing TNG parameters v2...
[2026-01-12 15:52:54,747 - INFO - convert_tng_to_spam.convert_collision_parameters_tng_to_spam_units]: Converting TNG units to SPAM units...
[2026-01-12 15:52:54,722 - INFO - convert_tng_to_spam._convert_tng_data_to_spam_units]: Converting TNG data to SPAM units for target: 81000000325459
[2026-01-12 15:52:54,756 - INFO - convert_tng_to_spam.__init__]: Target Initalized
[2026-01-12 15:52:54,770 - INFO - convert_tng_to_spam._transform_tng_orbits]: Generating TNG Orbits: 81000000325459
[2026-01-12 15:52:54,790 - INFO - convert_tng_to_spam._gen_spam_collision_parame

Failed:
Unable to open file (truncated file: eof = 180355072, sblock->base_addr = 0, stored_eof = 528464037)


[2026-01-12 15:52:57,062 - INFO - convert_tng_to_spam._gen_splines_hermite]: Interpolating path with Hermite cubic splines...
[2026-01-12 15:52:57,091 - INFO - convert_tng_to_spam._gen_moment_of_closest_approach]: Generating Most Recent Moment of Closest Approach (MOCA) Info...
[2026-01-12 15:52:57,150 - INFO - convert_tng_to_spam._gen_spam_collision_parameters]: Generating SPAM collision parameters for target: 81000000325459
[2026-01-12 15:52:57,175 - INFO - convert_tng_to_spam.standardize_collision_parameters_v2]: Standardizing TNG parameters v2...
[2026-01-12 15:52:57,226 - INFO - convert_tng_to_spam.convert_collision_parameters_tng_to_spam_units]: Converting TNG units to SPAM units...
[2026-01-12 15:52:57,246 - INFO - convert_tng_to_spam.__init__]: Target Initalized
[2026-01-12 15:52:57,258 - INFO - convert_tng_to_spam._gen_spam_collision_parameters]: Generating SPAM collision parameters for target: 81000000325459
[2026-01-12 15:52:57,278 - INFO - convert_tng_to_spam.standardize_co